# T06 + T07 — Cổng khả thi kỹ thuật trên Tesla T4

Notebook này chạy **cả hai task trong một phiên** để tiết kiệm quota GPU (30 giờ/tuần). Không chứa logic nào, chỉ lấy code, cài và gọi script.

**Notebook settings trước khi chạy:**

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens` — T07 **bắt buộc** cần, vì phải chạy trên mẫu ISE-DSC01 dài nhất
- Add-ons → Secrets: `HF_TOKEN` (không bắt buộc)

Không cần biết Kaggle gắn dataset vào đâu: script tự dò `/kaggle/input` tìm file `vihallu_train.csv`.

Chạy hết từ trên xuống rồi copy output của **ô 3, ô 4, ô 5 và ô 6** dán vào PR.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần: nếu thư mục đã có thì kéo bản mới về,
# vì `git clone` vào thư mục đã tồn tại sẽ hỏng và ta lặng lẽ chạy tiếp bằng code cũ.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: 9d2f91b Chốt float16 bỏ lớp 27, lưu hai biến thể lookback, bỏ token đầu (#13)


In [2]:
# Ô 2 — cài đặt. Không cài lại torch: image Kaggle đã có bản dựng theo đúng CUDA của máy.
!pip install -q --no-deps -e .
!pip install -q -U bitsandbytes accelerate transformers pytest

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
ERROR: Package 'vihallulens' requires a different Python: 3.12.13 not in '<3.12,>=3.11'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 47.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 106.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 34.4 MB/s eta 0:00:00


In [3]:
# Ô 3 — kiểm tra môi trường. Phải xanh trước khi chạy tiếp. Copy output dán vào PR.
# Chạy bằng tiến trình riêng chứ không import trong kernel: `pip install -e .` ghi một file
# .pth mà Python chỉ đọc lúc khởi động, nên kernel đang chạy sẵn có thể không thấy gói.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: đã nạp từ Kaggle Secrets")
except Exception:
    print("HF_TOKEN: không có, vẫn chạy được vì Qwen2.5 là mô hình mở")

get_ipython().system("python scripts/probe_env.py")

HF_TOKEN: không có, vẫn chạy được vì Qwen2.5 là mô hình mở

MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : 9d2f91b Chốt float16 bỏ lớp 27, lưu hai biến thể lookback, bỏ token đầu (#13)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.15.1
  bitsandbytes     : 0.50.1
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv


In [4]:
# Ô 4 — kiểm tra trên CPU: hook có nhận được attn_weights không, toán lookback có đúng không.
# Ô này hỏng thì DỪNG, đừng đốt quota GPU. Copy output dán vào PR.
!python -m pytest tests/test_attention_hook.py tests/test_attention_math.py -q
!python scripts/probe_attention_hook.py --tiny

.........................                                                [100%]

T07 — TRÍCH ATTENTION BẰNG FORWARD HOOK
  transformers          : 5.15.1
  chế độ                : tiny (CPU, trọng số ngẫu nhiên)
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 2.60MB/s]
tokenizer_config.json: 7.30kB [00:00, 18.0MB/s]
vocab.json: 2.78MB [00:00, 48.1MB/s]
merges.txt: 1.67MB [00:00, 109MB/s]
tokenizer.json: 7.03MB [00:00, 127MB/s]
  số lớp được hook      : 2

  mẫu tổng hợp
    ngữ cảnh              : 23 từ, 3 chunk
    lookback_per_chunk    : (2, 4, 9, 3)
    lookback_total        : (2, 4, 9)  (mẫu số cả prompt)
    lookback_context      : (2, 4, 9)  (mẫu số chỉ ngữ cảnh)
    self_attention        : (2, 4, 9)
    bị cắt ngữ cảnh       : False
    thời gian             : 55 ms
    VRAM đỉnh             : 0 MB
    tổng hàng attention   : 1.0000  (khỏe mạnh là 1,0000)
    lớp có nan/inf        : không có
    lookback_total        : trung bình 0.5000, min 0.4937, max 0.

In [5]:
# Ô 5 — T06. Copy output dán vào PR.
!python scripts/probe_load_model.py


NẠP MÔ HÌNH 4-BIT — Qwen/Qwen2.5-7B-Instruct
  torch                 : 2.10.0+cu128
  transformers          : 5.15.1
  GPU                   : Tesla T4
  Compute capability    : 7.5
  VRAM tổng             : 14,912 MB
model.safetensors.index.json: 27.8kB [00:00, 75.0MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [01:07<00:00, 16.96s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:07<00:00, 224MB/s]
generation_config.json: 100%|██████████████████| 243/243 [00:00<00:00, 1.07MB/s]

  Thời gian nạp         : 110.4 giây
  Lượng tử hóa          : nf4
  attn_implementation   : eager
  Số lớp / số đầu       : 28 / 28
  dtype tham số         : torch.float16

  VRAM đang cấp phát    : 5,302 MB
  VRAM đã đặt chỗ       : 5,462 MB
  VRAM đỉnh khi nạp     : 5,426 MB
  VRAM còn trống        : 9,450 MB   ← ngân sách cho attention

  Tiêu chí T06 (< 7,168 MB): ĐẠT


In [6]:
# Ô 6 — T07. Copy output dán vào PR.
!python scripts/probe_attention_hook.py


T07 — TRÍCH ATTENTION BẰNG FORWARD HOOK
  transformers          : 5.15.1
  chế độ                : Qwen/Qwen2.5-7B-Instruct
  thư mục dữ liệu       : /kaggle/input/datasets/unicorn1209/vihallulens
  compute dtype         : float16
  lớp bỏ qua            : [27]
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 22.66it/s]
  số lớp được hook      : 27

  ViHallu (~200 từ)
    ngữ cảnh              : 200 từ, 4 chunk
    lookback_per_chunk    : (27, 28, 47, 4)
    lookback_total        : (27, 28, 47)  (mẫu số cả prompt)
    lookback_context      : (27, 28, 47)  (mẫu số chỉ ngữ cảnh)
    self_attention        : (27, 28, 47)
    bị cắt ngữ cảnh       : False
    thời gian             : 951 ms
    VRAM đỉnh             : 5,496 MB
    tổng hàng attention   : 1.0000  (khỏe mạnh là 1,0000)
    lớp có nan/inf        : không có
    lookback_total        : trung bình 0.3613, min 0.0000, max 1.0000
    lookback_context      : trung bình 0.1969, min 0.0000, max 1.0000

  ISE-DSC0

In [7]:
# Ô 7 — chỉ chạy nếu ô 6 báo có lớp nan/inf.
# Qwen2.5 huấn luyện ở bfloat16; float16 có dải số hẹp hơn nhiều nên vài lớp có thể tràn
# thành inf, và softmax của inf ra nan. float32 không tràn nhưng tốn gấp đôi bộ nhớ.
# Mô hình đã tải sẵn trong phiên nên ô này chỉ mất khoảng 20 giây để nạp lại.
!python scripts/probe_attention_hook.py --compute-dtype float32


T07 — TRÍCH ATTENTION BẰNG FORWARD HOOK
  transformers          : 5.15.1
  chế độ                : Qwen/Qwen2.5-7B-Instruct
  thư mục dữ liệu       : /kaggle/input/datasets/unicorn1209/vihallulens
  compute dtype         : float32
  lớp bỏ qua            : [27]
Loading weights: 100%|████████████████████████| 339/339 [00:22<00:00, 15.34it/s]
  số lớp được hook      : 27

  ViHallu (~200 từ)
    ngữ cảnh              : 200 từ, 4 chunk
    lookback_per_chunk    : (27, 28, 47, 4)
    lookback_total        : (27, 28, 47)  (mẫu số cả prompt)
    lookback_context      : (27, 28, 47)  (mẫu số chỉ ngữ cảnh)
    self_attention        : (27, 28, 47)
    bị cắt ngữ cảnh       : False
    thời gian             : 2,116 ms
    VRAM đỉnh             : 7,747 MB
    tổng hàng attention   : 1.0000  (khỏe mạnh là 1,0000)
    lớp có nan/inf        : không có
    lookback_total        : trung bình 0.3614, min 0.0000, max 1.0000
    lookback_context      : trung bình 0.1969, min 0.0000, max 1.0000

  ISE-DS

In [8]:
# Ô 8 — so sánh hai kiểu số trên 20 mẫu trải từ ngắn tới dài, để chốt dùng float16 hay float32.
# Trả lời hai câu: có phải lúc nào cũng chỉ lớp 27 hỏng không, và các lớp còn sống ở float16
# có khớp float32 không. Khoảng 4-5 phút GPU. Copy toàn bộ output dán vào PR.
!python scripts/compare_dtypes.py --per-dataset 10


SO SÁNH float16 VỚI float32
  dữ liệu   : /kaggle/input/datasets/unicorn1209/vihallulens
  số mẫu    : 20
  độ dài    : 47 đến 4805 từ
Loading weights: 100%|████████████████████████| 339/339 [00:22<00:00, 15.33it/s]
    [float32] xong 20 mẫu trong 95 s
Loading weights: 100%|████████████████████████| 339/339 [00:14<00:00, 22.66it/s]
    [float16] xong 20 mẫu trong 24 s

  float32  :     4415 ms mỗi mẫu, VRAM đỉnh    10692 MB
  float16  :     1124 ms mỗi mẫu, VRAM đỉnh    11657 MB

  Mẫu có ít nhất một lớp nan ở fp16 : 20/20
  Tập hợp các lớp từng nan          : [27]

   Lớp |  mẫu nan (fp16) |  |Δ| trung bình |  |Δ| lớn nhất
  -----+-----------------+-----------------+--------------
     0 |       0/20      |         0.00568 |       0.91718
     1 |       0/20      |         0.00065 |       0.02490
     2 |       0/20      |         0.00061 |       0.02417
     3 |       0/20      |         0.00037 |       0.01123
     4 |       0/20      |         0.00062 |       0.01221
     5 |     

## Nếu ô 6 báo hết bộ nhớ

Đi theo bảng sáu nấc lùi ở mục 5 của `CLAUDE.md`, **theo thứ tự**, đừng nhảy cóc:

```python
# Nấc 1 — hạ ngân sách token. Đo ở T05: chỉ cắt thêm 1,09 % mẫu ISE-DSC01.
!python scripts/probe_attention_hook.py --max-context-tokens 2048

# Nấc 4 — lùi mô hình, cùng họ nên không đổi dòng code nào.
!python scripts/probe_attention_hook.py --model Qwen/Qwen2.5-3B-Instruct
```

Nấc 2 và nấc 3 cần sửa code, nấc 3 là đổi kiến trúc nên phải hỏi trước. Ghi lại nấc nào đã thử vào **Nhật ký chặn** cuối `TASKS.md`.